# Step 2 - Training the three models

Multi-label toxicity classification (6 labels) on the cleaned, anonymised comments
(`processed_data/train_clean.csv`). Three families are trained and compared on the same split:

| Model | Family | Notes |
|---|---|---|
| `lstm` | statistical / neural | Bidirectional LSTM trained from scratch (PyTorch, CUDA) |
| `bert` | transformer | Fine-tuned `bert-base-uncased` (Hugging Face, CUDA, mixed precision) |
| `laya` | decision model | Fine-tuned [Laya](https://huggingface.co/convaiinnovations/laya): non-autoregressive decision model, one yes/no question per label (CUDA) |

Every model reports precision, recall, F1 and ROC-AUC per label, plus training and inference time.
Decision thresholds are tuned on the validation split (labels are very imbalanced), never on test.

**CUDA is required**: the notebook stops if no GPU is visible.

## 0. Setup

In [23]:
import gc
import json
import logging
import math
import random
import time
from collections import Counter
from collections.abc import Callable, Iterator, Sequence
from pathlib import Path
from typing import Any

import laya
import numpy as np
import pandas as pd
import torch
from laya.train import TrainConfig, finetune
from numpy.typing import NDArray
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch import Tensor, nn
from torch.amp.grad_scaler import GradScaler
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True
)
logger = logging.getLogger("train")

# Set to a small number (e.g. 5000) for a quick smoke test, None for the full run.
LIMIT: int | None = None

SEED = 42
VALIDATION_SIZE = 0.1
TEXT_COLUMN = "comment_text"
LABELS: tuple[str, ...] = (
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
)

FloatArray = NDArray[np.float32]
IntArray = NDArray[np.int64]


def find_project_root() -> Path:
    """Walk up from the working directory until pyproject.toml is found."""
    for parent in (Path.cwd(), *Path.cwd().parents):
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("pyproject.toml not found above the working directory")


PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "raw_data"
PROCESSED_DATA_DIR = PROJECT_ROOT / "processed_data"
MODELS_DIR = PROJECT_ROOT / "models"

## 1. Helpers: CUDA check, data loading, thresholds and metrics

Thresholds maximise F1 per label on the validation split. `finalize` evaluates on the test split
(rows Kaggle scores, i.e. labels different from `-1`) and writes `models/<name>/metrics.json`.

In [24]:
def set_seed(seed: int = SEED) -> None:
    """Seed every random number generator used in this notebook."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def require_cuda() -> torch.device:
    """Return the CUDA device, or fail loudly: training on CPU is not supported here."""
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. Check the driver with `nvidia-smi` and make sure the "
            "CUDA build of PyTorch is installed (`uv sync`, index pytorch-cu128)."
        )
    logger.info("Using CUDA device: %s", torch.cuda.get_device_name(0))
    return torch.device("cuda")


def read_clean(path: Path) -> pd.DataFrame:
    """Read a cleaned CSV; comments that became empty are read back as NaN."""
    df = pd.read_csv(path)
    df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)
    return df


def load_train_val(limit: int | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load the cleaned training set and split it into train and validation parts."""
    df = read_clean(PROCESSED_DATA_DIR / "train_clean.csv")
    if limit is not None:
        df = df.head(limit)
    train_df, val_df = train_test_split(
        df, test_size=VALIDATION_SIZE, random_state=SEED, stratify=df["toxic"]
    )
    logger.info("Train rows: %d, validation rows: %d", len(train_df), len(val_df))
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def load_test(limit: int | None = None) -> pd.DataFrame:
    """Load the cleaned test set joined with its labels (rows labelled -1 are not scored)."""
    df = read_clean(PROCESSED_DATA_DIR / "test_clean.csv")
    df = df.drop(columns=list(LABELS), errors="ignore")
    labels = pd.read_csv(RAW_DATA_DIR / "test_labels.csv")
    merged = df.merge(labels, on="id", how="inner")
    merged = merged[merged["toxic"] != -1].reset_index(drop=True)
    if limit is not None:
        merged = merged.head(limit)
    logger.info("Scored test rows: %d", len(merged))
    return merged


def labels_array(df: pd.DataFrame) -> IntArray:
    """Return the label columns as an (n_rows, 6) integer array."""
    return df[list(LABELS)].to_numpy(dtype=np.int64)


def mean_auc(y_true: IntArray, y_prob: FloatArray) -> float:
    """Mean ROC-AUC over the labels (early stopping criterion)."""
    return float(np.mean([roc_auc_score(y_true[:, i], y_prob[:, i]) for i in range(len(LABELS))]))


def find_thresholds(y_true: IntArray, y_prob: FloatArray) -> list[float]:
    """Pick, per label, the threshold that maximises F1 (default 0.5 gives poor recall)."""
    thresholds: list[float] = []
    for i in range(len(LABELS)):
        precision, recall, candidates = precision_recall_curve(y_true[:, i], y_prob[:, i])
        f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
        thresholds.append(float(candidates[int(np.argmax(f1))]) if len(candidates) else 0.5)
    return thresholds


def evaluate(y_true: IntArray, y_prob: FloatArray, thresholds: list[float]) -> dict[str, Any]:
    """Precision, recall, F1 and ROC-AUC per label, plus their macro average."""
    per_label: dict[str, dict[str, float]] = {}
    for i, label in enumerate(LABELS):
        predicted = (y_prob[:, i] >= thresholds[i]).astype(np.int64)
        precision = float(precision_score(y_true[:, i], predicted, zero_division=0))
        recall = float(recall_score(y_true[:, i], predicted, zero_division=0))
        f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
        per_label[label] = {
            "threshold": thresholds[i],
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "roc_auc": float(roc_auc_score(y_true[:, i], y_prob[:, i])),
        }
    macro = {
        metric: float(np.mean([scores[metric] for scores in per_label.values()]))
        for metric in ("precision", "recall", "f1", "roc_auc")
    }
    return {"per_label": per_label, "macro": macro}


def finalize(
    name: str,
    y_val: IntArray,
    val_prob: FloatArray,
    y_test: IntArray,
    test_prob: FloatArray,
    train_seconds: float,
    inference_seconds: float,
    config: dict[str, Any],
) -> dict[str, Any]:
    """Tune thresholds on validation, evaluate on test and write metrics.json."""
    thresholds = find_thresholds(y_val, val_prob)
    results: dict[str, Any] = evaluate(y_test, test_prob, thresholds)
    results["model"] = name
    results["train_seconds"] = round(train_seconds, 1)
    results["inference_seconds"] = round(inference_seconds, 2)
    results["inference_ms_per_comment"] = round(1000 * inference_seconds / len(y_test), 4)
    results["test_rows"] = len(y_test)
    results["config"] = config
    out_dir = MODELS_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "metrics.json").write_text(
        json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    macro = results["macro"]
    logger.info(
        "[%s] macro precision=%.4f recall=%.4f f1=%.4f roc_auc=%.4f | train %.0fs | "
        "inference %.3f ms/comment",
        name,
        macro["precision"],
        macro["recall"],
        macro["f1"],
        macro["roc_auc"],
        train_seconds,
        results["inference_ms_per_comment"],
    )
    return results

## 2. Load the data and check CUDA

In [25]:
set_seed()
device = require_cuda()

train_df, val_df = load_train_val(LIMIT)
test_df = load_test(LIMIT)
y_train = labels_array(train_df)
y_val = labels_array(val_df)
y_test = labels_array(test_df)

pd.DataFrame(
    {
        "train": y_train.mean(axis=0),
        "validation": y_val.mean(axis=0),
        "test": y_test.mean(axis=0),
    },
    index=list(LABELS),
).round(4)

2026-11-10 15:30:48,646 INFO train: Using CUDA device: NVIDIA GeForce RTX 3050 6GB Laptop GPU
2026-11-10 15:30:51,154 INFO train: Train rows: 143613, validation rows: 15958
2026-11-10 15:30:53,118 INFO train: Scored test rows: 63978


,train,validation,test
toxic,0.0958,0.0958,0.0952
severe_toxic,0.0101,0.0093,0.0057
obscene,0.0529,0.0531,0.0577
threat,0.0030,0.0031,0.0033
insult,0.0493,0.0501,0.0536
identity_hate,0.0087,0.0096,0.0111


## 3. Model 1 - BiLSTM (statistical / neural)

Tokens are the whitespace-separated words of the cleaned text (already lower-cased, no punctuation).
The vocabulary keeps the 50,000 most frequent words seen at least 3 times; the others map to `<unk>`.
The network is `Embedding -> BiLSTM -> [max pooling + mean pooling] -> Linear(6)`, trained with a
binary cross-entropy loss per label and mixed precision on the GPU.

In [26]:
LSTM_CONFIG: dict[str, Any] = {
    "max_vocab": 50_000,
    "min_freq": 3,
    "max_len": 200,
    "embed_dim": 128,
    "hidden_dim": 128,
    "dropout": 0.3,
    "epochs": 5,
    "batch_size": 256,
    "lr": 2e-3,
}
PAD_ID, UNK_ID = 0, 1


def build_vocab(texts: Sequence[str], max_vocab: int, min_freq: int) -> dict[str, int]:
    """Map the most frequent words to integer ids (0 = padding, 1 = unknown)."""
    counts = Counter(token for text in texts for token in text.split())
    vocab = {"<pad>": PAD_ID, "<unk>": UNK_ID}
    for token, count in counts.most_common(max_vocab - len(vocab)):
        if count < min_freq:
            break
        vocab[token] = len(vocab)
    return vocab


def encode_texts(
    texts: Sequence[str], vocab: dict[str, int], max_len: int
) -> tuple[Tensor, Tensor]:
    """Return padded token ids and true lengths (an empty comment counts as one unknown token)."""
    ids = torch.zeros((len(texts), max_len), dtype=torch.long)
    lengths = torch.ones(len(texts), dtype=torch.long)
    for row, text in enumerate(texts):
        tokens = [vocab.get(token, UNK_ID) for token in text.split()[:max_len]] or [UNK_ID]
        ids[row, : len(tokens)] = torch.tensor(tokens, dtype=torch.long)
        lengths[row] = len(tokens)
    return ids, lengths


class BiLstmClassifier(nn.Module):
    """Embedding -> BiLSTM -> max + mean pooling over real tokens -> 6 logits."""

    def __init__(
        self, vocab_size: int, embed_dim: int, hidden_dim: int, num_labels: int, dropout: float
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 4, num_labels)

    def forward(self, ids: Tensor, lengths: Tensor) -> Tensor:
        embedded = self.dropout(self.embedding(ids))
        packed = pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(output, batch_first=True, total_length=ids.size(1))
        positions = torch.arange(ids.size(1), device=ids.device).unsqueeze(0)
        mask = (positions < lengths.unsqueeze(1)).unsqueeze(-1)
        max_pool = output.masked_fill(~mask, float("-inf")).max(dim=1).values
        mean_pool = (output * mask).sum(dim=1) / lengths.unsqueeze(1)
        logits: Tensor = self.classifier(self.dropout(torch.cat([max_pool, mean_pool], dim=1)))
        return logits


def predict_lstm(
    model: BiLstmClassifier, ids: Tensor, lengths: Tensor, batch_size: int
) -> FloatArray:
    """Probabilities for every label, shape (n_rows, 6)."""
    model.eval()
    outputs: list[FloatArray] = []
    with torch.no_grad():
        for start in range(0, len(ids), batch_size):
            batch_ids = ids[start : start + batch_size].to(device)
            batch_lengths = lengths[start : start + batch_size].to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(batch_ids, batch_lengths)
            outputs.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(outputs).astype(np.float32)


def fit_lstm(
    model: BiLstmClassifier,
    train_data: tuple[Tensor, Tensor, IntArray],
    val_data: tuple[Tensor, Tensor, IntArray],
    config: dict[str, Any],
) -> BiLstmClassifier:
    """Train with AdamW + mixed precision; keep the epoch with the best validation ROC-AUC."""
    train_ids, train_lengths, train_labels = train_data
    val_ids, val_lengths, val_labels = val_data
    dataset = torch.utils.data.TensorDataset(
        train_ids, train_lengths, torch.from_numpy(train_labels).float()
    )
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=config["batch_size"], shuffle=True, pin_memory=True
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"])
    scaler = GradScaler("cuda")
    loss_fn = nn.BCEWithLogitsLoss()
    best_auc = -1.0
    best_state = {n: t.detach().cpu().clone() for n, t in model.state_dict().items()}
    for epoch in range(config["epochs"]):
        model.train()
        total_loss = 0.0
        for batch_ids, batch_lengths, batch_labels in loader:
            batch_ids = batch_ids.to(device, non_blocking=True)
            batch_lengths = batch_lengths.to(device, non_blocking=True)
            batch_labels = batch_labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(batch_ids, batch_lengths)
            loss = loss_fn(logits.float(), batch_labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.item())
        val_auc = mean_auc(val_labels, predict_lstm(model, val_ids, val_lengths, 1024))
        logger.info(
            "LSTM epoch %d/%d: train loss %.4f, validation ROC-AUC %.4f",
            epoch + 1,
            config["epochs"],
            total_loss / len(loader),
            val_auc,
        )
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {n: t.detach().cpu().clone() for n, t in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

In [6]:
set_seed()
start = time.perf_counter()
max_len = LSTM_CONFIG["max_len"]
vocab = build_vocab(
    train_df[TEXT_COLUMN].tolist(), LSTM_CONFIG["max_vocab"], LSTM_CONFIG["min_freq"]
)
logger.info("LSTM vocabulary size: %d", len(vocab))
train_ids, train_lengths = encode_texts(train_df[TEXT_COLUMN].tolist(), vocab, max_len)
val_ids, val_lengths = encode_texts(val_df[TEXT_COLUMN].tolist(), vocab, max_len)

lstm_model = BiLstmClassifier(
    len(vocab),
    LSTM_CONFIG["embed_dim"],
    LSTM_CONFIG["hidden_dim"],
    len(LABELS),
    LSTM_CONFIG["dropout"],
).to(device)
lstm_model = fit_lstm(
    lstm_model, (train_ids, train_lengths, y_train), (val_ids, val_lengths, y_val), LSTM_CONFIG
)
lstm_train_seconds = time.perf_counter() - start

lstm_val_prob = predict_lstm(lstm_model, val_ids, val_lengths, 1024)
start = time.perf_counter()  # inference time includes encoding the raw text
test_ids, test_lengths = encode_texts(test_df[TEXT_COLUMN].tolist(), vocab, max_len)
lstm_test_prob = predict_lstm(lstm_model, test_ids, test_lengths, 1024)
torch.cuda.synchronize()
lstm_inference_seconds = time.perf_counter() - start

lstm_results = finalize(
    "lstm", y_val, lstm_val_prob, y_test, lstm_test_prob,
    lstm_train_seconds, lstm_inference_seconds, LSTM_CONFIG,
)
lstm_dir = MODELS_DIR / "lstm"
torch.save(lstm_model.state_dict(), lstm_dir / "model.pt")
(lstm_dir / "vocab.json").write_text(json.dumps(vocab), encoding="utf-8")

del lstm_model
torch.cuda.empty_cache()

2026-11-10 11:33:39,838 INFO train: LSTM vocabulary size: 49978
C:\Users\houeg\AppData\Local\Temp\ipykernel_21224\611522123.py:92: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  train_ids, train_lengths, torch.from_numpy(train_labels).float()
2026-11-10 11:35:24,247 INFO train: LSTM epoch 1/5: train loss 0.0875, validation ROC-AUC 0.9655
2026-11-10 11:36:43,338 INFO train: LSTM epoch 2/5: train loss 0.0544, validation ROC-AUC 0.9755
2026-11-10 11:38:00,423 INFO train: LSTM epoch 3/5: train loss 0.0472, validation ROC-AUC 0.9802
2026-11-10 11:39:31,621 INFO train: LSTM epoch 4/5:

## 4. Model 2 - BERT (transformer)

`bert-base-uncased` fine-tuned for multi-label classification (the cleaned text is lower-cased, so the
uncased model fits). Sequences are truncated to 128 tokens and padded per batch. The batch size of 16
with mixed precision is chosen for a 6 GB GPU (RTX 3050); lower `batch_size` or `max_len` if memory runs
out, or set `model_name` to `distilbert-base-uncased` for a lighter and faster run.

In [27]:
BERT_CONFIG: dict[str, Any] = {
    "model_name": "bert-base-uncased",
    "max_len": 128,
    "epochs": 2,
    "batch_size": 16,
    "eval_batch_size": 64,
    "lr": 2e-5,
    "warmup_ratio": 0.06,
}


def iter_batches(
    input_ids: list[list[int]],
    labels: IntArray | None,
    batch_size: int,
    pad_id: int,
    rng: np.random.Generator | None,
) -> Iterator[tuple[Tensor, Tensor, Tensor | None]]:
    """Yield padded (input_ids, attention_mask, labels) batches; shuffled when rng is given."""
    order = rng.permutation(len(input_ids)) if rng is not None else np.arange(len(input_ids))
    for start in range(0, len(order), batch_size):
        indices = order[start : start + batch_size]
        longest = max(len(input_ids[i]) for i in indices)
        ids = torch.full((len(indices), longest), pad_id, dtype=torch.long)
        mask = torch.zeros((len(indices), longest), dtype=torch.long)
        for row, i in enumerate(indices):
            ids[row, : len(input_ids[i])] = torch.tensor(input_ids[i], dtype=torch.long)
            mask[row, : len(input_ids[i])] = 1
        batch_labels = None if labels is None else torch.from_numpy(labels[indices]).float()
        yield ids, mask, batch_labels


def predict_bert(
    model: Any, input_ids: list[list[int]], pad_id: int, batch_size: int
) -> FloatArray:
    """Probabilities for every label, shape (n_rows, 6), in the input order."""
    model.eval()
    outputs: list[FloatArray] = []
    with torch.no_grad():
        for ids, mask, _ in iter_batches(input_ids, None, batch_size, pad_id, None):
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            outputs.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(outputs).astype(np.float32)


def fit_bert(
    model: Any,
    train_ids: list[list[int]],
    train_labels: IntArray,
    val_ids: list[list[int]],
    val_labels: IntArray,
    pad_id: int,
    config: dict[str, Any],
    save_dir: Path,
) -> None:
    """Fine-tune with AdamW, linear warm-up/decay and mixed precision; save the best epoch."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=0.01)
    steps_per_epoch = math.ceil(len(train_ids) / config["batch_size"])
    total_steps = steps_per_epoch * config["epochs"]
    scheduler = get_linear_schedule_with_warmup(  # type: ignore[no-untyped-call]
        optimizer, int(config["warmup_ratio"] * total_steps), total_steps
    )
    scaler = GradScaler("cuda")
    loss_fn = nn.BCEWithLogitsLoss()
    best_auc = -1.0
    for epoch in range(config["epochs"]):
        model.train()
        rng = np.random.default_rng(SEED + epoch)
        running_loss = 0.0
        batches = iter_batches(train_ids, train_labels, config["batch_size"], pad_id, rng)
        for step, (ids, mask, batch_labels) in enumerate(batches, start=1):
            assert batch_labels is not None
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            loss = loss_fn(logits.float(), batch_labels.to(device))
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += float(loss.item())
            if step % 500 == 0:
                logger.info(
                    "BERT epoch %d step %d/%d: loss %.4f",
                    epoch + 1, step, steps_per_epoch, running_loss / step,
                )
        val_prob = predict_bert(model, val_ids, pad_id, config["eval_batch_size"])
        val_auc = mean_auc(val_labels, val_prob)
        logger.info(
            "BERT epoch %d/%d: validation ROC-AUC %.4f", epoch + 1, config["epochs"], val_auc
        )
        if val_auc > best_auc:
            best_auc = val_auc
            model.save_pretrained(save_dir)

In [6]:
set_seed()
bert_dir = MODELS_DIR / "bert"
bert_dir.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(BERT_CONFIG["model_name"])
pad_id = int(tokenizer.pad_token_id)


def tokenize(texts: list[str]) -> list[list[int]]:
    """Token ids truncated to max_len (special tokens included, no padding yet)."""
    encoded = tokenizer(texts, truncation=True, max_length=BERT_CONFIG["max_len"])
    return [list(ids) for ids in encoded["input_ids"]]


start = time.perf_counter()
bert_train_ids = tokenize(train_df[TEXT_COLUMN].tolist())
bert_val_ids = tokenize(val_df[TEXT_COLUMN].tolist())

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_CONFIG["model_name"], num_labels=len(LABELS), problem_type="multi_label_classification"
).to(device)
fit_bert(bert_model, bert_train_ids, y_train, bert_val_ids, y_val, pad_id, BERT_CONFIG, bert_dir)
bert_train_seconds = time.perf_counter() - start

# Reload the best epoch saved on disk.
bert_model = AutoModelForSequenceClassification.from_pretrained(bert_dir).to(device)
tokenizer.save_pretrained(bert_dir)
bert_val_prob = predict_bert(bert_model, bert_val_ids, pad_id, BERT_CONFIG["eval_batch_size"])

start = time.perf_counter()  # inference time includes tokenisation of the raw text
bert_test_ids = tokenize(test_df[TEXT_COLUMN].tolist())
bert_test_prob = predict_bert(bert_model, bert_test_ids, pad_id, BERT_CONFIG["eval_batch_size"])
torch.cuda.synchronize()
bert_inference_seconds = time.perf_counter() - start

bert_results = finalize(
    "bert", y_val, bert_val_prob, y_test, bert_test_prob,
    bert_train_seconds, bert_inference_seconds, BERT_CONFIG,
)

del bert_model
torch.cuda.empty_cache()

2026-11-10 12:16:20,473 INFO httpx2: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-11-10 12:16:20,592 INFO httpx2: HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 Temporary Redirect"
2026-11-10 12:16:20,706 INFO httpx2: HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased "HTTP/1.1 200 OK"
2026-11-10 12:16:20,848 INFO httpx2: HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/86b5e0934494bd15c9632b12f734a8a67f723594/config.json "HTTP/1.1 200 OK"
2026-11-10 12:16:20,849 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-11-10 12:16:20,970 INFO httpx2: HTTP Request: GET https://huggingface.co/bert-base-uncased/resolve/86b5e0934494bd15c9632b12f734a8a67f723594/config.json "HTTP/1.1 200 OK"
c:\Users\houeg\projects\tp1_digital_social_score\.venv\L

## 5. Model 3 - Laya (decision model)

[Laya](https://huggingface.co/convaiinnovations/laya) is a non-autoregressive "System 1" decision model: given a
state (here the cleaned comment) and typed questions, it returns typed answers with calibrated probabilities in a
single forward pass. It never generates text. Each Jigsaw label becomes one yes/no question (`noul` type) and the
answer is `P(true)`, which gives the six probabilities the other models produce.

The base checkpoints are close to chance on a new task (see the Laya README), so the model is **fine-tuned** with
`laya.train.finetune` (RLCD loss, then temperature calibration on a held-out slice that Laya keeps by itself).

Choices made for a 6 GB GPU (RTX 3050):

| Choice | Value | Why |
|---|---|---|
| Base checkpoint | `multilingual` (mmBERT-base, 322M) | Lighter and about 2.2x faster than the English ModernBERT-large (421M, alias `english`) |
| `freeze_encoder` | `True` | Only the decision head is trained. The Laya guide targets 16 GB cards for full fine-tuning; set it to `False` on a bigger GPU for better accuracy |
| Training subset | all comments with at least one label + half as many clean ones | Bounds the training time (one decision per label and per comment); the rare labels keep all their examples |
| `epochs` | 1 | Shortens the run; raise it (e.g. 3) if the metrics are not good enough |
| Thresholds | tuned on the untouched validation split | The balanced training subset shifts the base rate, the thresholds absorb it |
| `max_len` | 256 tokens | Saves memory; about 5% of the comments are longer and get truncated |

Known caveat from the README: a `noul` question can follow its own wording instead of the text. Fine-tuning on
labelled data is the mitigation, which is why the base checkpoint is never used as is.

In [28]:
LAYA_CONFIG: dict[str, Any] = {
    "base": "multilingual",
    "freeze_encoder": True,
    "epochs": 1,
    "micro_batch": 4,
    "grad_accum": 16,
    "max_len": 256,
    "clean_ratio": 0.5,
    "predict_batch_size": 64,
    # Comments used to evaluate Laya (None = all). Laya scores 6 rows per comment, so inference is
    # slow: use e.g. 8_000 / 20_000 for a quick estimate, None for the final full run.
    "eval_val_rows": None,
    "eval_test_rows": None,
}
LAYA_RETRAIN = False  # True ignores an existing checkpoint and trains again
LAYA_DIR = MODELS_DIR / "laya"
LAYA_TRAIN_FILE = PROCESSED_DATA_DIR / "laya" / "train.jsonl"

# One yes/no ("noul") question per label; the answer is P(true).
LAYA_QUESTIONS: dict[str, dict[str, Any]] = {
    "toxic": {
        "type": "noul",
        "instructions": "Is the comment toxic: rude, disrespectful or likely to push someone out "
        "of the discussion?",
    },
    "severe_toxic": {
        "type": "noul",
        "instructions": "Is the comment severely toxic: very hateful, aggressive or disrespectful?",
    },
    "obscene": {
        "type": "noul",
        "instructions": "Does the comment contain obscene or vulgar language?",
    },
    "threat": {
        "type": "noul",
        "instructions": "Does the comment contain a threat of violence or harm?",
    },
    "insult": {
        "type": "noul",
        "instructions": "Does the comment insult a person or a group?",
    },
    "identity_hate": {
        "type": "noul",
        "instructions": "Does the comment attack someone because of their identity (origin, "
        "religion, gender, sexual orientation, disability)?",
    },
}


def laya_state(text: str) -> str:
    """Laya input: the cleaned comment, or a placeholder when cleaning left nothing."""
    return text if text.strip() else "(empty comment)"


def write_laya_training_file(
    df: pd.DataFrame, labels: IntArray, clean_ratio: float, path: Path
) -> int:
    """Write the JSONL training file and return the number of comments it holds.

    Keeps every comment with at least one label plus ``clean_ratio`` times as many clean ones.
    """
    has_label = labels.max(axis=1) == 1
    positives = np.flatnonzero(has_label)
    clean = np.flatnonzero(~has_label)
    rng = np.random.default_rng(SEED)
    n_clean = min(len(clean), int(len(positives) * clean_ratio))
    kept_clean = rng.choice(clean, size=n_clean, replace=False)
    selected = rng.permutation(np.concatenate([positives, kept_clean]))
    texts = df[TEXT_COLUMN].tolist()
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for index in selected:
            row = {
                "state": laya_state(texts[index]),
                "questions": LAYA_QUESTIONS,
                "expected": {label: bool(labels[index, i]) for i, label in enumerate(LABELS)},
            }
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    return int(len(selected))


def predict_laya(agent: laya.Agent, texts: list[str], batch_size: int) -> FloatArray:
    """P(true) of each label's question, shape (n_rows, 6), in the input order."""
    results = agent.predict_batch(
        [laya_state(text) for text in texts],
        LAYA_QUESTIONS,
        batch_size=batch_size,
        sort_by_length=True,
    )
    return np.array(
        [[float(result["answers"][label]["noul"]) for label in LABELS] for result in results],
        dtype=np.float32,
    )

In [20]:
set_seed()
laya_time_file = LAYA_DIR / "train_seconds.json"
if (LAYA_DIR / "rl_agent_config.json").exists() and not LAYA_RETRAIN:
    logger.info("Reusing the fine-tuned Laya checkpoint in %s", LAYA_DIR)
    laya_train_seconds = (
        float(json.loads(laya_time_file.read_text(encoding="utf-8"))["train_seconds"])
        if laya_time_file.exists()
        else 0.0
    )
else:
    laya_rows = write_laya_training_file(
        train_df, y_train, LAYA_CONFIG["clean_ratio"], LAYA_TRAIN_FILE
    )
    logger.info("Laya training file: %d comments, %d decisions", laya_rows, laya_rows * len(LABELS))
    start = time.perf_counter()
    laya_report = finetune(
        str(LAYA_TRAIN_FILE),
        LAYA_CONFIG["base"],
        str(LAYA_DIR),
        TrainConfig(
            epochs=LAYA_CONFIG["epochs"],
            micro_batch=LAYA_CONFIG["micro_batch"],
            grad_accum=LAYA_CONFIG["grad_accum"],
            max_len=LAYA_CONFIG["max_len"],
            freeze_encoder=LAYA_CONFIG["freeze_encoder"],
            seed=SEED,
        ),
        device="cuda",
    )
    laya_train_seconds = time.perf_counter() - start
    laya_time_file.write_text(json.dumps({"train_seconds": laya_train_seconds}), encoding="utf-8")
    logger.info(
        "Laya fine-tuning summary: %s",
        {k: v for k, v in laya_report.items() if isinstance(v, int | float | str)},
    )
    gc.collect()
    torch.cuda.empty_cache()

2026-11-10 14:22:46,374 INFO train: Laya training file: 21933 comments, 131598 decisions
2026-11-10 14:22:46,776 INFO httpx2: HTTP Request: GET https://huggingface.co/api/models/convaiinnovations/laya/revision/main "HTTP/1.1 200 OK"
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 375.80it/s]


train items 131198, calibration items 400, eval items 400 (calibration_slice), skipped {}, device cuda
optimizer budget: 131198 train item(s), effective batch up to 64 (micro_batch 4 x grad_accum 16), 2050 optimizer updates over 1 epoch(s)
epoch 1/1 step 100 loss 0.0449
epoch 1/1 step 200 loss -0.3073
epoch 1/1 step 300 loss -0.0407
epoch 1/1 step 400 loss 0.1714
epoch 1/1 step 500 loss 0.1827
epoch 1/1 step 600 loss 1.4769
epoch 1/1 step 700 loss 0.1519
epoch 1/1 step 800 loss 0.6390
epoch 1/1 step 900 loss 0.3199
epoch 1/1 step 1000 loss -0.1733
epoch 1/1 step 1100 loss -0.0613
epoch 1/1 step 1200 loss 0.4798
epoch 1/1 step 1300 loss 1.2160
epoch 1/1 step 1400 loss 0.4303
epoch 1/1 step 1500 loss 0.6332
epoch 1/1 step 1600 loss 0.6284
epoch 1/1 step 1700 loss 0.2790
epoch 1/1 step 1800 loss 0.0739
epoch 1/1 step 1900 loss 0.6496
epoch 1/1 step 2000 loss 0.7781
epoch 1/1 step 2100 loss 2.9302
epoch 1/1 step 2200 loss 1.0966
epoch 1/1 step 2300 loss -0.3885
epoch 1/1 step 2400 loss -0.

C:\Users\houeg\AppData\Local\Temp\ipykernel_37148\1088461554.py:16: RuntimeWarning: laya.train: noul calibration: the fitted temperature 5 is on the [0.5, 5] clamp
  laya_report = finetune(



=== Calibration Evidence (in-sample calibration slice, 400 items; not held-out) ===
Metric                 Before         After          Delta         
-----------------------------------------------------------------
Accuracy               0.8450         0.9075         +0.0625       
Loss                   0.3637         0.2737         -0.0900       
ECE                    0.0621         0.0499         -0.0122       
Brier (multi-class)    0.2194         0.1544         -0.0650       
Brier (top-1)          0.1097         0.0772         -0.0325       
Mean Confidence        0.8975         0.9468         +0.0493       
-----------------------------------------------------------------



2026-11-10 15:04:16,996 INFO train: Laya fine-tuning summary: {'train_items': 131198, 'calibration_items': 400, 'eval_items': 400, 'output_dir': 'c:\\Users\\houeg\\projects\\tp1_digital_social_score\\models\\laya'}


### Laya inference speed

Laya scores one row per question, so a comment costs about 6 forward passes (6 labels) and evaluating the whole
validation and test sets is slow. The next cell times 512 comments for three batch sizes and extrapolates the
duration of the full evaluation, so you can decide before launching it:

- pick the fastest `batch_size` and set `LAYA_CONFIG["predict_batch_size"]` accordingly;
- if the full evaluation is still too long, set `LAYA_CONFIG["eval_val_rows"]` and `eval_test_rows` (e.g. 8,000 and
  20,000). Rare labels (`threat`) then have few positives, so the metrics are noisier; the number of evaluated rows is
  stored in `metrics.json` as `test_rows`.

The cell also frees the GPU memory left behind by an interrupted evaluation (`free_gpu_memory`).

In [29]:
def free_gpu_memory() -> None:
    """Release the Laya agent left behind by an interrupted cell, then empty the CUDA cache."""
    globals().pop("laya_agent", None)
    gc.collect()
    torch.cuda.empty_cache()


def evaluation_size(total: int, limit: int | None) -> int:
    """Number of comments scored for a split, given the optional row limit."""
    return total if limit is None else min(total, limit)


def benchmark_laya(
    texts: list[str], batch_sizes: tuple[int, ...], n_eval_comments: int
) -> pd.DataFrame:
    """Time Laya on a few comments for several batch sizes and extrapolate to the evaluation."""
    free_gpu_memory()
    agent = laya.Agent(str(LAYA_DIR), device="cuda")
    rows: list[dict[str, Any]] = []
    try:
        predict_laya(agent, texts[:32], 16)  # warm-up: CUDA kernels and caches
        for batch_size in batch_sizes:
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            start = time.perf_counter()
            predict_laya(agent, texts, batch_size)
            torch.cuda.synchronize()
            ms_per_comment = 1000 * (time.perf_counter() - start) / len(texts)
            rows.append(
                {
                    "batch_size": batch_size,
                    "ms_per_comment": round(ms_per_comment, 2),
                    "full_evaluation_minutes": round(ms_per_comment * n_eval_comments / 60_000, 1),
                    "peak_vram_mb": round(torch.cuda.max_memory_allocated() / 2**20),
                }
            )
    finally:
        del agent
        free_gpu_memory()
    return pd.DataFrame(rows).set_index("batch_size")


n_eval_comments = evaluation_size(len(val_df), LAYA_CONFIG["eval_val_rows"]) + evaluation_size(
    len(test_df), LAYA_CONFIG["eval_test_rows"]
)
logger.info("Laya evaluation covers %d comments", n_eval_comments)
benchmark_laya(val_df[TEXT_COLUMN].tolist()[:512], (16, 32, 64), n_eval_comments)

2026-11-10 15:31:24,130 INFO train: Laya evaluation covers 79936 comments


KeyboardInterrupt: 

In [30]:
def subsample_index(n_rows: int, limit: int | None) -> IntArray:
    """Sorted random row indices (fixed seed), or every row when no limit applies."""
    if limit is None or limit >= n_rows:
        return np.arange(n_rows, dtype=np.int64)
    chosen = np.random.default_rng(SEED).choice(n_rows, size=limit, replace=False)
    return np.sort(chosen).astype(np.int64)


laya_val_index = subsample_index(len(val_df), LAYA_CONFIG["eval_val_rows"])
laya_test_index = subsample_index(len(test_df), LAYA_CONFIG["eval_test_rows"])
laya_val_texts = val_df[TEXT_COLUMN].iloc[laya_val_index].tolist()
laya_test_texts = test_df[TEXT_COLUMN].iloc[laya_test_index].tolist()
laya_val_labels = y_val[laya_val_index]
laya_test_labels = y_test[laya_test_index]
logger.info(
    "Laya evaluation on %d validation and %d test comments",
    len(laya_val_texts),
    len(laya_test_texts),
)

free_gpu_memory()
laya_agent = laya.Agent(str(LAYA_DIR), device="cuda")
batch_size = LAYA_CONFIG["predict_batch_size"]
try:
    laya_val_prob = predict_laya(laya_agent, laya_val_texts, batch_size)

    start = time.perf_counter()  # inference time covers the whole scored test set
    laya_test_prob = predict_laya(laya_agent, laya_test_texts, batch_size)
    torch.cuda.synchronize()
    laya_inference_seconds = time.perf_counter() - start
finally:
    free_gpu_memory()

laya_results = finalize(
    "laya", laya_val_labels, laya_val_prob, laya_test_labels, laya_test_prob,
    laya_train_seconds, laya_inference_seconds, LAYA_CONFIG,
)

2026-11-10 15:33:39,897 INFO train: Laya evaluation on 15958 validation and 63978 test comments


KeyboardInterrupt: 

## 6. Comparison

Macro-averaged metrics on the scored test rows, plus training time and inference cost.

In [ ]:
rows: list[dict[str, Any]] = []
for model_name in ("lstm", "bert", "laya"):
    metrics_path = MODELS_DIR / model_name / "metrics.json"
    if not metrics_path.exists():
        continue
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    rows.append(
        {
            "model": model_name,
            **metrics["macro"],
            "train_seconds": metrics["train_seconds"],
            "inference_ms_per_comment": metrics["inference_ms_per_comment"],
            "test_rows": metrics["test_rows"],
        }
    )
pd.DataFrame(rows).set_index("model").round(4)

In [ ]:
per_label: dict[str, pd.DataFrame] = {}
for model_name in ("lstm", "bert", "laya"):
    metrics_path = MODELS_DIR / model_name / "metrics.json"
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        per_label[model_name] = pd.DataFrame(metrics["per_label"]).T[
            ["precision", "recall", "f1", "roc_auc"]
        ]
pd.concat(per_label, names=["model", "label"]).round(4)

### Single-comment latency

Time needed to score **one comment** (batch size 1) with each trained model, as an API receives requests one at a
time. Each model is reloaded from `models/<name>/`, then called repeatedly on the same comment:

- the comment is already cleaned; anonymisation (spaCy + regex) is not included, it is a separate step of the API;
- each timing includes encoding or tokenisation and the transfer of the probabilities back to the CPU, and excludes
  loading the model;
- the first `LATENCY_WARMUP` calls are discarded (CUDA kernels, caches), then `LATENCY_RUNS` calls are timed;
- `p50` is the typical latency, `p95` the slow tail (one call in twenty is slower than this).

These are **GPU** timings measured in the notebook. On a CPU server such as a Vertex AI endpoint the figures will
differ, so measure again there. Models whose folder is missing are skipped.

In [31]:
SINGLE_COMMENT = "you are a stupid idiot and nobody cares about your edits"
LATENCY_WARMUP = 10
LATENCY_RUNS = 100


def latency_stats(call: Callable[[], object]) -> dict[str, float]:
    """Latency in ms of repeated single-comment calls (GPU synchronised after each call)."""
    for _ in range(LATENCY_WARMUP):
        call()
    torch.cuda.synchronize()
    timings: list[float] = []
    for _ in range(LATENCY_RUNS):
        start = time.perf_counter()
        call()
        torch.cuda.synchronize()
        timings.append(1000 * (time.perf_counter() - start))
    ordered = sorted(timings)
    return {
        "p50_ms": ordered[len(ordered) // 2],
        "p95_ms": ordered[int(0.95 * len(ordered))],
        "mean_ms": sum(ordered) / len(ordered),
    }


def measure_lstm() -> dict[str, float]:
    lstm_dir = MODELS_DIR / "lstm"
    words = json.loads((lstm_dir / "vocab.json").read_text(encoding="utf-8"))
    model = BiLstmClassifier(
        len(words),
        LSTM_CONFIG["embed_dim"],
        LSTM_CONFIG["hidden_dim"],
        len(LABELS),
        LSTM_CONFIG["dropout"],
    ).to(device)
    model.load_state_dict(torch.load(lstm_dir / "model.pt", map_location=device, weights_only=True))
    model.eval()

    def call() -> FloatArray:
        ids, lengths = encode_texts([SINGLE_COMMENT], words, LSTM_CONFIG["max_len"])
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(ids.to(device), lengths.to(device))
        return np.asarray(torch.sigmoid(logits.float()).cpu().numpy(), dtype=np.float32)

    return latency_stats(call)


def measure_bert() -> dict[str, float]:
    bert_path = MODELS_DIR / "bert"
    bert_tokenizer = AutoTokenizer.from_pretrained(bert_path)
    model = AutoModelForSequenceClassification.from_pretrained(bert_path).to(device)
    model.eval()

    def call() -> FloatArray:
        batch = bert_tokenizer(
            [SINGLE_COMMENT],
            truncation=True,
            max_length=BERT_CONFIG["max_len"],
            return_tensors="pt",
        ).to(device)
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(**batch).logits
        return np.asarray(torch.sigmoid(logits.float()).cpu().numpy(), dtype=np.float32)

    return latency_stats(call)


def measure_laya() -> dict[str, float]:
    agent = laya.Agent(str(LAYA_DIR), device="cuda")

    def call() -> dict[str, Any]:
        result: dict[str, Any] = agent.system_one(laya_state(SINGLE_COMMENT), LAYA_QUESTIONS)
        return result

    return latency_stats(call)


latency_rows: list[dict[str, Any]] = []
for model_name, measure in (
    ("lstm", measure_lstm),
    ("bert", measure_bert),
    ("laya", measure_laya),
):
    if not (MODELS_DIR / model_name).exists():
        logger.warning("Skipping %s: no folder in %s", model_name, MODELS_DIR)
        continue
    latency_rows.append({"model": model_name, **measure()})
    gc.collect()
    torch.cuda.empty_cache()
pd.DataFrame(latency_rows).set_index("model").round(2)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 367.73it/s]


,p50_ms,p95_ms,mean_ms
model,,,
lstm,2.91,10.52,4.12
bert,30.41,42.66,30.67
laya,40.53,75.63,45.77


## 7. Digital Social Score (0-100)

The API must return one number from 0 to 100 per text. It is a **weighted combination of the six label
probabilities**, with weights that reflect how serious each label is:

`score = 100 x sum(weight_label x probability_label)`, weights summing to 1.

| Label | Weight | Why |
|---|---:|---|
| `toxic` | 0.40 | General label, true for almost every toxic comment, so it carries the base level |
| `severe_toxic` | 0.20 | Extreme toxicity |
| `threat` | 0.15 | Threats are a serious harm even when rare |
| `identity_hate` | 0.10 | Attacks on a group identity |
| `insult` | 0.10 | Personal attack |
| `obscene` | 0.05 | Crude language, the least harmful on its own |

The weights are a documented design choice, not learned values: adjust them in `SCORE_WEIGHTS` if the
policy changes. The probabilities come straight from each model (sigmoid outputs, or Laya's temperature-scaled
probabilities), so the score is a ranking of severity rather than a calibrated probability. The evaluation below checks that the score separates clean, toxic and severely toxic
comments on the test set.

In [22]:
SCORE_WEIGHTS: dict[str, float] = {
    "toxic": 0.40,
    "severe_toxic": 0.20,
    "obscene": 0.05,
    "threat": 0.15,
    "insult": 0.10,
    "identity_hate": 0.10,
}
if set(SCORE_WEIGHTS) != set(LABELS) or abs(sum(SCORE_WEIGHTS.values()) - 1.0) > 1e-9:
    raise ValueError("SCORE_WEIGHTS must cover every label and sum to 1")


def toxicity_score(probs: FloatArray) -> FloatArray:
    """Digital Social Score in [0, 100] from label probabilities of shape (n_rows, 6)."""
    weights = np.array([SCORE_WEIGHTS[label] for label in LABELS], dtype=np.float32)
    return np.asarray(100.0 * (probs @ weights), dtype=np.float32)


MODELS_DIR.mkdir(parents=True, exist_ok=True)
(MODELS_DIR / "score_weights.json").write_text(
    json.dumps(SCORE_WEIGHTS, indent=2), encoding="utf-8"
)

119

In [ ]:
# For each model: the variable with its test probabilities and the one with the matching labels
# (Laya may have been evaluated on a subsample of the test set).
test_probabilities = {
    "lstm": ("lstm_test_prob", "y_test"),
    "bert": ("bert_test_prob", "y_test"),
    "laya": ("laya_test_prob", "laya_test_labels"),
}
available = {
    name: (globals()[probs_var], globals()[labels_var])
    for name, (probs_var, labels_var) in test_probabilities.items()
    if probs_var in globals() and labels_var in globals()
}
if not available:
    logger.warning("No test probabilities in memory: run the model cells above first")

severe_index = LABELS.index("severe_toxic")
score_rows: list[dict[str, Any]] = []
for model_name, (probs, labels) in available.items():
    scores = toxicity_score(probs)
    any_label = labels.max(axis=1)
    is_severe = labels[:, severe_index] == 1
    score_rows.append(
        {
            "model": model_name,
            "roc_auc_vs_any_label": roc_auc_score(any_label, scores),
            "mean_score_clean": float(scores[any_label == 0].mean()),
            "mean_score_toxic": float(scores[any_label == 1].mean()),
            "mean_score_severe_toxic": float(scores[is_severe].mean()),
        }
    )
pd.DataFrame(score_rows).set_index("model").round(3)